# Experiment 05 — MLP on Engineered Features

## Purpose and decision rules

Histogram gradient boosting is the current development leader:

**mean subject-fold macro-F1: 0.7446 ± 0.0432**

This experiment asks whether a compact neural network can learn feature
interactions from the same 222 engineered watch accelerometer–gyroscope
features more effectively than the tree-based models.

The input representation, development examples, activity labels, and five
outer subject folds remain unchanged. This isolates the learning algorithm:
the MLP does not receive raw time-series samples or any additional features.

### Fixed architecture

The predeclared MLP is:

    222 standardized features
    → Linear(222, 256)
    → ReLU
    → Dropout(0.30)
    → Linear(256, 128)
    → ReLU
    → Dropout(0.30)
    → Linear(128, 18)

No architecture search, class weighting, batch normalization, or feature
selection is performed.

### Nested training protocol

For every outer subject fold:

1. reserve the outer fold exclusively for evaluation;
2. use the next numbered fold as inner validation;
3. use the remaining three folds for inner training;
4. fit feature standardization only on the inner-training examples;
5. select the training duration using inner-validation macro-F1;
6. reinitialize the MLP;
7. refit standardization on all outer-training examples;
8. train for the selected number of epochs;
9. evaluate once on the untouched outer-fold subjects.

The fixed training configuration uses cross-entropy loss, AdamW with learning
rate 0.001 and weight decay 0.0001, batch size 256, a maximum of 100 inner
epochs, patience of 10 epochs, and a minimum qualifying macro-F1 improvement of
0.001.

### Decision rules

Seed 42 is evaluated first.

The MLP replaces histogram gradient boosting only if it:

- reaches at least **0.7546 mean macro-F1**, a gain of at least 0.01;
- improves at least 3 of the 5 outer folds.

Because MLP training is stochastic, seeds 43 and 44 will be evaluated only if
the seed-42 result is competitive, defined in advance as mean macro-F1 of at
least **0.7346**, within 0.01 of the current leader.

The reserved test set remains untouched.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [
            CURRENT_DIR,
            *CURRENT_DIR.parents,
        ]
        if (
            candidate
            / "modeling"
            / "artifacts"
            / "engineered_222_meta.json"
        ).exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the activity_project root."
    )

MODEL_ARTIFACT_DIR = (
    PROJECT_ROOT
    / "modeling"
    / "artifacts"
)

FEATURE_PATH = (
    MODEL_ARTIFACT_DIR
    / "engineered_222_train.parquet"
)

FEATURE_META_PATH = (
    MODEL_ARTIFACT_DIR
    / "engineered_222_meta.json"
)

PREPROCESSING_META_PATH = (
    PROJECT_ROOT
    / "modified"
    / "watch_pair_20hz"
    / "preprocessing_meta.json"
)

TRAIN_METADATA_PATH = (
    PROJECT_ROOT
    / "modified"
    / "watch_pair_20hz"
    / "train_metadata.parquet"
)

DEVELOPMENT_FOLDS_PATH = (
    PROJECT_ROOT
    / "baseline"
    / "artifacts"
    / "development_folds.json"
)

feature_meta = json.loads(
    FEATURE_META_PATH.read_text(
        encoding="utf-8"
    )
)

preprocessing_meta = json.loads(
    PREPROCESSING_META_PATH.read_text(
        encoding="utf-8"
    )
)

development_folds = json.loads(
    DEVELOPMENT_FOLDS_PATH.read_text(
        encoding="utf-8"
    )
)

feature_table = pd.read_parquet(
    FEATURE_PATH
)

train_metadata = pd.read_parquet(
    TRAIN_METADATA_PATH
)

feature_names = feature_meta[
    "feature_names"
]

assert feature_table.columns.tolist() == [
    "array_index",
    *feature_names,
]

assert np.array_equal(
    feature_table[
        "array_index"
    ].to_numpy(),
    train_metadata[
        "array_index"
    ].to_numpy(),
)

X_features = (
    feature_table[
        feature_names
    ]
    .to_numpy(
        dtype=np.float32
    )
)

ACTIVITY_CODES = preprocessing_meta[
    "labels"
]["activity_codes_in_order"]

ACTIVITY_TO_INDEX = {
    activity_code: activity_index
    for activity_index, activity_code
    in enumerate(ACTIVITY_CODES)
}

labels = (
    train_metadata[
        "activity_code"
    ]
    .astype(str)
    .map(ACTIVITY_TO_INDEX)
    .to_numpy(dtype=np.int64)
)

fold_assignments = (
    train_metadata[
        "fold"
    ]
    .astype(int)
    .to_numpy()
)

subject_ids = (
    train_metadata[
        "user_id"
    ].to_numpy()
)

expected_fold_by_subject = {
    int(subject_id): int(fold)
    for fold, subjects in (
        development_folds[
            "validation_subject_ids_by_fold"
        ].items()
    )
    for subject_id in subjects
}

observed_fold_by_subject = {
    int(row.user_id): int(row.fold)
    for row in (
        train_metadata[
            ["user_id", "fold"]
        ]
        .drop_duplicates()
        .itertuples(index=False)
    )
}

assert X_features.shape == (
    25_585,
    222,
)
assert np.isfinite(
    X_features
).all()
assert observed_fold_by_subject == (
    expected_fold_by_subject
)

DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
)

print(
    f"Examples: "
    f"{len(train_metadata):,}"
)
print(
    "Feature shape:",
    X_features.shape,
)
print(
    "Feature dtype:",
    X_features.dtype,
)
print(
    "Activities:",
    len(ACTIVITY_CODES),
)
print(
    "Subjects:",
    np.unique(
        subject_ids
    ).size,
)
print(
    "Training device:",
    DEVICE,
)

Examples: 25,585
Feature shape: (25585, 222)
Feature dtype: float32
Activities: 18
Subjects: 41
Training device: mps


## 1. Nested subject splits and feature standardization

The same deterministic inner-fold rotation used by the CNN experiment is
retained:

| Outer fold | Inner-validation fold |
|---:|---:|
| 1 | 2 |
| 2 | 3 |
| 3 | 4 |
| 4 | 5 |
| 5 | 1 |

For each outer fold, the other three folds form the inner-training partition.
All partitions are subject-disjoint.

Feature means and standard deviations are learned separately for each training
partition. No global standardizer is fitted, and neither inner-validation nor
outer-validation examples contribute to the corresponding normalization
parameters.

In [2]:
from sklearn.preprocessing import (
    StandardScaler,
)


INNER_VALIDATION_FOLD_BY_OUTER = {
    1: 2,
    2: 3,
    3: 4,
    4: 5,
    5: 1,
}

nested_split_masks = {}
split_rows = []

for outer_fold, inner_validation_fold in (
    INNER_VALIDATION_FOLD_BY_OUTER.items()
):
    outer_validation_mask = (
        fold_assignments
        == outer_fold
    )

    outer_training_mask = (
        ~outer_validation_mask
    )

    inner_validation_mask = (
        fold_assignments
        == inner_validation_fold
    )

    inner_training_mask = (
        outer_training_mask
        & ~inner_validation_mask
    )

    inner_training_subjects = set(
        subject_ids[
            inner_training_mask
        ]
    )

    inner_validation_subjects = set(
        subject_ids[
            inner_validation_mask
        ]
    )

    outer_validation_subjects = set(
        subject_ids[
            outer_validation_mask
        ]
    )

    assert inner_training_subjects.isdisjoint(
        inner_validation_subjects
    )
    assert inner_training_subjects.isdisjoint(
        outer_validation_subjects
    )
    assert inner_validation_subjects.isdisjoint(
        outer_validation_subjects
    )

    nested_split_masks[
        outer_fold
    ] = {
        "outer_training": (
            outer_training_mask
        ),
        "outer_validation": (
            outer_validation_mask
        ),
        "inner_training": (
            inner_training_mask
        ),
        "inner_validation": (
            inner_validation_mask
        ),
    }

    split_rows.append({
        "outer_fold": outer_fold,
        "inner_validation_fold": (
            inner_validation_fold
        ),
        "inner_training_subjects": len(
            inner_training_subjects
        ),
        "inner_validation_subjects": len(
            inner_validation_subjects
        ),
        "outer_validation_subjects": len(
            outer_validation_subjects
        ),
        "inner_training_examples": int(
            inner_training_mask.sum()
        ),
        "inner_validation_examples": int(
            inner_validation_mask.sum()
        ),
        "outer_validation_examples": int(
            outer_validation_mask.sum()
        ),
        "inner_training_activities": int(
            np.unique(
                labels[
                    inner_training_mask
                ]
            ).size
        ),
        "inner_validation_activities": int(
            np.unique(
                labels[
                    inner_validation_mask
                ]
            ).size
        ),
    })

nested_split_overview = pd.DataFrame(
    split_rows
)


def fit_feature_standardizer(
    training_mask,
):
    standardizer = StandardScaler()

    standardizer.fit(
        X_features[
            training_mask
        ]
    )

    assert standardizer.n_features_in_ == (
        X_features.shape[1]
    )

    return standardizer


def transform_features(
    standardizer,
    selection_mask,
):
    return (
        standardizer
        .transform(
            X_features[
                selection_mask
            ]
        )
        .astype(
            np.float32,
            copy=False,
        )
    )


display(
    nested_split_overview
)

print(
    "Nested subject splits and "
    "fold-specific standardization ready."
)

,outer_fold,inner_validation_fold,inner_training_subjects,inner_validation_subjects,outer_validation_subjects,inner_training_examples,inner_validation_examples,outer_validation_examples,inner_training_activities,inner_validation_activities
0,1,2,24,8,9,14945,4970,5670,18,18
1,2,3,25,8,8,15645,4970,4970,18,18
2,3,4,25,8,8,15645,4970,4970,18,18
3,4,5,25,8,8,15610,5005,4970,18,18
4,5,1,24,9,8,14910,5670,5005,18,18


Nested subject splits and fold-specific standardization ready.


## 2. Feature dataset and fixed MLP architecture

For each training run, the standardizer is fitted on that run's training
partition. The transformed feature values, encoded activity label, and original
array index are then placed in a PyTorch dataset.

The array index is retained so that outer-fold predictions can later be
reassembled in exact canonical row order.

The MLP contains two hidden layers with ReLU activations and 0.30 dropout. It
does not use batch normalization because the input features are already
standardized using training-partition statistics. Dropout is active only during
training.

In [3]:
from torch import nn
from torch.utils.data import (
    DataLoader,
    TensorDataset,
)


BATCH_SIZE = 256
INITIAL_SEED = 42


def make_feature_loader(
    selection_mask,
    standardizer,
    *,
    shuffle,
    seed,
):
    selected_indices = np.flatnonzero(
        selection_mask
    )

    transformed_features = (
        standardizer
        .transform(
            X_features[
                selected_indices
            ]
        )
        .astype(
            np.float32,
            copy=False,
        )
    )

    selected_labels = labels[
        selected_indices
    ]

    dataset = TensorDataset(
        torch.from_numpy(
            transformed_features
        ),
        torch.from_numpy(
            selected_labels
        ),
        torch.from_numpy(
            selected_indices.astype(
                np.int64
            )
        ),
    )

    generator = (
        torch.Generator()
        .manual_seed(seed)
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator,
        num_workers=0,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
    )


class FeatureMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                X_features.shape[1],
                256,
            ),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(
                256,
                128,
            ),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(
                128,
                len(ACTIVITY_CODES),
            ),
        )

    def forward(self, features):
        return self.network(
            features
        )


preview_masks = nested_split_masks[1]

preview_standardizer = (
    fit_feature_standardizer(
        preview_masks[
            "inner_training"
        ]
    )
)

preview_loader = make_feature_loader(
    preview_masks[
        "inner_validation"
    ],
    preview_standardizer,
    shuffle=False,
    seed=INITIAL_SEED,
)

(
    preview_features,
    preview_labels,
    preview_indices,
) = next(iter(preview_loader))

preview_model = FeatureMLP()

with torch.no_grad():
    preview_logits = preview_model(
        preview_features
    )

parameter_count = sum(
    parameter.numel()
    for parameter
    in preview_model.parameters()
    if parameter.requires_grad
)

assert preview_features.shape == (
    len(preview_features),
    222,
)
assert preview_logits.shape == (
    len(preview_features),
    18,
)
assert torch.isfinite(
    preview_features
).all()
assert torch.isfinite(
    preview_logits
).all()

print(
    "Trainable parameters:",
    f"{parameter_count:,}",
)
print(
    "Feature batch shape:",
    tuple(
        preview_features.shape
    ),
)
print(
    "Label batch shape:",
    tuple(
        preview_labels.shape
    ),
)
print(
    "Logit batch shape:",
    tuple(
        preview_logits.shape
    ),
)
print(
    "Array-index batch shape:",
    tuple(
        preview_indices.shape
    ),
)

Trainable parameters: 92,306
Feature batch shape: (256, 222)
Label batch shape: (256,)
Logit batch shape: (256, 18)
Array-index batch shape: (256,)


## 3. Fixed training and inner epoch selection

The MLP is trained with the fixed configuration declared at the start of the
notebook. Inner-validation macro-F1 is measured after every epoch.

An epoch is selected when its macro-F1 exceeds the previous selected value by
more than 0.001. Training stops after ten consecutive epochs without such an
improvement, or after 100 epochs.

The inner model itself is not retained. Only its selected training duration is
used later when a newly initialized model is fitted on all outer-training
subjects.

In [4]:
from sklearn.metrics import (
    f1_score,
)


LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_INNER_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 0.001


def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if DEVICE.type == "mps":
        torch.mps.manual_seed(seed)

    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)


def train_one_epoch(
    model,
    data_loader,
    optimizer,
    loss_function,
):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for (
        features,
        targets,
        _,
    ) in data_loader:
        features = features.to(
            DEVICE
        )
        targets = targets.to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(features)

        loss = loss_function(
            logits,
            targets,
        )

        loss.backward()
        optimizer.step()

        batch_size = len(targets)

        total_loss += (
            float(
                loss.detach().cpu()
            )
            * batch_size
        )

        total_examples += (
            batch_size
        )

    return (
        total_loss
        / total_examples
    )


def predict_data_loader(
    model,
    data_loader,
):
    model.eval()

    true_parts = []
    predicted_parts = []
    index_parts = []

    with torch.no_grad():
        for (
            features,
            targets,
            array_indices,
        ) in data_loader:
            features = features.to(
                DEVICE
            )

            logits = model(features)

            predicted = logits.argmax(
                dim=1
            )

            true_parts.append(
                targets.numpy()
            )

            predicted_parts.append(
                predicted
                .cpu()
                .numpy()
            )

            index_parts.append(
                array_indices.numpy()
            )

    return {
        "y_true": np.concatenate(
            true_parts
        ),
        "y_pred": np.concatenate(
            predicted_parts
        ),
        "array_index": np.concatenate(
            index_parts
        ),
    }


def calculate_macro_f1(
    y_true,
    y_pred,
):
    return f1_score(
        y_true,
        y_pred,
        labels=np.arange(
            len(ACTIVITY_CODES)
        ),
        average="macro",
        zero_division=0,
    )


def select_epoch_with_inner_validation(
    outer_fold,
    seed,
    *,
    verbose=True,
):
    masks = nested_split_masks[
        outer_fold
    ]

    standardizer = (
        fit_feature_standardizer(
            masks[
                "inner_training"
            ]
        )
    )

    training_loader = (
        make_feature_loader(
            masks[
                "inner_training"
            ],
            standardizer,
            shuffle=True,
            seed=(
                seed
                + outer_fold * 100
            ),
        )
    )

    validation_loader = (
        make_feature_loader(
            masks[
                "inner_validation"
            ],
            standardizer,
            shuffle=False,
            seed=seed,
        )
    )

    set_training_seed(seed)

    model = FeatureMLP().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    loss_function = (
        nn.CrossEntropyLoss()
    )

    history_rows = []

    best_macro_f1 = -np.inf
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(
        1,
        MAX_INNER_EPOCHS + 1,
    ):
        training_loss = (
            train_one_epoch(
                model,
                training_loader,
                optimizer,
                loss_function,
            )
        )

        validation_predictions = (
            predict_data_loader(
                model,
                validation_loader,
            )
        )

        validation_macro_f1 = (
            calculate_macro_f1(
                validation_predictions[
                    "y_true"
                ],
                validation_predictions[
                    "y_pred"
                ],
            )
        )

        improved = (
            validation_macro_f1
            > best_macro_f1
            + EARLY_STOPPING_MIN_DELTA
        )

        if improved:
            best_macro_f1 = (
                validation_macro_f1
            )

            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history_rows.append({
            "epoch": epoch,
            "training_loss": (
                training_loss
            ),
            "inner_validation_macro_f1": (
                validation_macro_f1
            ),
            "selected_improvement": (
                improved
            ),
        })

        if verbose and (
            improved
            or epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Epoch {epoch:03d}: "
                f"loss={training_loss:.4f}, "
                f"inner macro-F1="
                f"{validation_macro_f1:.4f}"
            )

        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            break

    result = {
        "outer_fold": outer_fold,
        "inner_validation_fold": (
            INNER_VALIDATION_FOLD_BY_OUTER[
                outer_fold
            ]
        ),
        "seed": seed,
        "selected_epoch": int(
            best_epoch
        ),
        "best_inner_macro_f1": float(
            best_macro_f1
        ),
        "epochs_run": len(
            history_rows
        ),
        "history": pd.DataFrame(
            history_rows
        ),
    }

    del model
    del optimizer

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result

First real inner epoch-selection run:

In [5]:
inner_selection_cache = {}

first_selection_key = (
    1,
    INITIAL_SEED,
)

first_selection = (
    select_epoch_with_inner_validation(
        outer_fold=1,
        seed=INITIAL_SEED,
    )
)

inner_selection_cache[
    first_selection_key
] = first_selection

display(
    pd.Series({
        "Outer fold": 1,
        "Inner-validation fold": (
            first_selection[
                "inner_validation_fold"
            ]
        ),
        "Selected epoch": (
            first_selection[
                "selected_epoch"
            ]
        ),
        "Best inner macro-F1": (
            first_selection[
                "best_inner_macro_f1"
            ]
        ),
        "Epochs run": (
            first_selection[
                "epochs_run"
            ]
        ),
    }).to_frame("value")
)

display(
    first_selection[
        "history"
    ].tail(12).round(4)
)

Epoch 001: loss=1.7013, inner macro-F1=0.5898
Epoch 002: loss=0.9981, inner macro-F1=0.6375
Epoch 003: loss=0.8582, inner macro-F1=0.6489
Epoch 004: loss=0.7784, inner macro-F1=0.6583
Epoch 005: loss=0.7173, inner macro-F1=0.6626
Epoch 008: loss=0.6185, inner macro-F1=0.6665
Epoch 009: loss=0.5967, inner macro-F1=0.6763
Epoch 011: loss=0.5502, inner macro-F1=0.6837
Epoch 020: loss=0.4392, inner macro-F1=0.6867
Epoch 021: loss=0.4318, inner macro-F1=0.6880
Epoch 031: loss=0.3566, inner macro-F1=0.6690


,value
Outer fold,1.000000
Inner-validation fold,2.000000
Selected epoch,21.000000
Best inner macro-F1,0.687959
Epochs run,31.000000


,epoch,training_loss,inner_validation_macro_f1,selected_improvement
19,20,0.4392,0.6867,True
20,21,0.4318,0.6880,True
21,22,0.4199,0.6811,False
22,23,0.4092,0.6786,False
23,24,0.3984,0.6876,False
24,25,0.3976,0.6769,False
25,26,0.3891,0.6804,False
26,27,0.3842,0.6824,False
27,28,0.3753,0.6831,False
28,29,0.3697,0.6858,False


## 4. Seed-42 nested outer-fold evaluation

For each outer fold, the inner split selects the training duration. The MLP is
then reinitialized, and a new standardizer is fitted using all outer-training
subjects. The model is trained for exactly the selected number of epochs before
being evaluated once on the untouched outer-validation subjects.

Outer-fold performance does not affect normalization, training duration, or
any other model decision.

In [6]:
def train_outer_model(
    outer_fold,
    seed,
    selected_epochs,
):
    masks = nested_split_masks[
        outer_fold
    ]

    standardizer = (
        fit_feature_standardizer(
            masks[
                "outer_training"
            ]
        )
    )

    training_loader = (
        make_feature_loader(
            masks[
                "outer_training"
            ],
            standardizer,
            shuffle=True,
            seed=(
                seed
                + outer_fold * 1_000
            ),
        )
    )

    validation_loader = (
        make_feature_loader(
            masks[
                "outer_validation"
            ],
            standardizer,
            shuffle=False,
            seed=seed,
        )
    )

    set_training_seed(seed)

    model = FeatureMLP().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    loss_function = (
        nn.CrossEntropyLoss()
    )

    training_losses = []

    for epoch in range(
        1,
        selected_epochs + 1,
    ):
        training_loss = (
            train_one_epoch(
                model,
                training_loader,
                optimizer,
                loss_function,
            )
        )

        training_losses.append(
            training_loss
        )

    predictions = predict_data_loader(
        model,
        validation_loader,
    )

    expected_indices = np.flatnonzero(
        masks[
            "outer_validation"
        ]
    )

    assert set(
        predictions[
            "array_index"
        ]
    ) == set(expected_indices)

    prediction_table = pd.DataFrame({
        "array_index": (
            predictions[
                "array_index"
            ]
        ),
        "outer_fold": outer_fold,
        "y_true_index": (
            predictions[
                "y_true"
            ]
        ),
        "y_pred_index": (
            predictions[
                "y_pred"
            ]
        ),
    })

    result = {
        "outer_fold": outer_fold,
        "seed": seed,
        "selected_epochs": (
            selected_epochs
        ),
        "final_training_loss": float(
            training_losses[-1]
        ),
        "prediction_table": (
            prediction_table
        ),
    }

    del model
    del optimizer

    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result

In [7]:
mlp_outer_results = {}
mlp_prediction_parts = []

selection_rows = []
outer_training_rows = []

for outer_fold in sorted(
    nested_split_masks
):
    cache_key = (
        outer_fold,
        INITIAL_SEED,
    )

    if (
        cache_key
        not in inner_selection_cache
    ):
        inner_selection_cache[
            cache_key
        ] = (
            select_epoch_with_inner_validation(
                outer_fold=outer_fold,
                seed=INITIAL_SEED,
                verbose=False,
            )
        )

    selection = (
        inner_selection_cache[
            cache_key
        ]
    )

    print(
        f"Fold {outer_fold}: "
        f"selected epoch "
        f"{selection['selected_epoch']} "
        f"(inner macro-F1 "
        f"{selection['best_inner_macro_f1']:.4f})"
    )

    outer_result = train_outer_model(
        outer_fold=outer_fold,
        seed=INITIAL_SEED,
        selected_epochs=(
            selection[
                "selected_epoch"
            ]
        ),
    )

    mlp_outer_results[
        cache_key
    ] = outer_result

    mlp_prediction_parts.append(
        outer_result[
            "prediction_table"
        ]
    )

    selection_rows.append({
        "outer_fold": outer_fold,
        "inner_validation_fold": (
            selection[
                "inner_validation_fold"
            ]
        ),
        "selected_epoch": (
            selection[
                "selected_epoch"
            ]
        ),
        "best_inner_macro_f1": (
            selection[
                "best_inner_macro_f1"
            ]
        ),
        "epochs_run": (
            selection[
                "epochs_run"
            ]
        ),
    })

    outer_training_rows.append({
        "outer_fold": outer_fold,
        "selected_epochs": (
            outer_result[
                "selected_epochs"
            ]
        ),
        "final_training_loss": (
            outer_result[
                "final_training_loss"
            ]
        ),
    })

mlp_inner_selection_summary = (
    pd.DataFrame(
        selection_rows
    )
)

mlp_outer_training_summary = (
    pd.DataFrame(
        outer_training_rows
    )
)

mlp_oof_predictions = (
    pd.concat(
        mlp_prediction_parts,
        ignore_index=True,
    )
    .sort_values("array_index")
    .reset_index(drop=True)
)

assert len(
    mlp_oof_predictions
) == len(train_metadata)

assert mlp_oof_predictions[
    "array_index"
].is_unique

assert np.array_equal(
    mlp_oof_predictions[
        "array_index"
    ].to_numpy(),
    np.arange(len(train_metadata)),
)

display(
    mlp_inner_selection_summary.round(4)
)

display(
    mlp_outer_training_summary.round(4)
)

print(
    "Complete seed-42 out-of-fold "
    "predictions constructed."
)

Fold 1: selected epoch 21 (inner macro-F1 0.6880)
Fold 2: selected epoch 36 (inner macro-F1 0.7933)
Fold 3: selected epoch 28 (inner macro-F1 0.7050)
Fold 4: selected epoch 19 (inner macro-F1 0.7406)
Fold 5: selected epoch 21 (inner macro-F1 0.7718)


,outer_fold,inner_validation_fold,selected_epoch,best_inner_macro_f1,epochs_run
0,1,2,21,0.6880,31
1,2,3,36,0.7933,46
2,3,4,28,0.7050,38
3,4,5,19,0.7406,29
4,5,1,21,0.7718,31


,outer_fold,selected_epochs,final_training_loss
0,1,21,0.4522
1,2,36,0.3450
2,3,28,0.4000
3,4,19,0.4346
4,5,21,0.4365


Complete seed-42 out-of-fold predictions constructed.


## 5. Seed-42 performance and continuation decision

Seed-42 out-of-fold performance is calculated separately for each outer subject
fold and then averaged across the five folds.

Two decisions are evaluated:

1. whether the MLP is competitive enough to justify repeating seeds 43 and 44;
2. whether it passes the stronger replacement gate against histogram gradient
   boosting.

The competitiveness threshold is mean macro-F1 of 0.7346. The replacement gate
requires mean macro-F1 of at least 0.7546 and improvement over histogram
gradient boosting on at least three outer folds.

In [8]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
)


HGB_MEAN_MACRO_F1 = 0.7446

HGB_FOLD_MACRO_F1 = {
    1: 0.7852,
    2: 0.6895,
    3: 0.7887,
    4: 0.7159,
    5: 0.7434,
}

MLP_COMPETITIVE_THRESHOLD = 0.7346
MLP_REPLACEMENT_TARGET = 0.7546
MIN_FOLDS_IMPROVED = 3


def calculate_classification_metrics(
    y_true,
    y_pred,
):
    class_indices = np.arange(
        len(ACTIVITY_CODES)
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": recall_score(
            y_true,
            y_pred,
            labels=class_indices,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=class_indices,
            average="macro",
            zero_division=0,
        ),
    }


mlp_fold_rows = []

for outer_fold, fold_predictions in (
    mlp_oof_predictions.groupby(
        "outer_fold",
        sort=True,
    )
):
    fold_metrics = (
        calculate_classification_metrics(
            fold_predictions[
                "y_true_index"
            ],
            fold_predictions[
                "y_pred_index"
            ],
        )
    )

    mlp_fold_rows.append({
        "outer_fold": int(
            outer_fold
        ),
        **fold_metrics,
        "hgb_macro_f1": (
            HGB_FOLD_MACRO_F1[
                int(outer_fold)
            ]
        ),
    })

mlp_fold_metrics = pd.DataFrame(
    mlp_fold_rows
)

mlp_fold_metrics[
    "mlp_minus_hgb"
] = (
    mlp_fold_metrics[
        "macro_f1"
    ]
    - mlp_fold_metrics[
        "hgb_macro_f1"
    ]
)

mlp_metric_summary = (
    mlp_fold_metrics[
        [
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
    .agg(["mean", "std"])
)

mlp_mean_macro_f1 = (
    mlp_metric_summary.loc[
        "mean",
        "macro_f1",
    ]
)

mlp_folds_better_than_hgb = int(
    mlp_fold_metrics[
        "mlp_minus_hgb"
    ].gt(0).sum()
)

mlp_is_competitive = bool(
    mlp_mean_macro_f1
    >= MLP_COMPETITIVE_THRESHOLD
)

mlp_passes_replacement_gate = bool(
    mlp_mean_macro_f1
    >= MLP_REPLACEMENT_TARGET
    and mlp_folds_better_than_hgb
    >= MIN_FOLDS_IMPROVED
)

mlp_summary = pd.Series({
    "accuracy_mean": (
        mlp_metric_summary.loc[
            "mean",
            "accuracy",
        ]
    ),
    "accuracy_fold_std": (
        mlp_metric_summary.loc[
            "std",
            "accuracy",
        ]
    ),
    "balanced_accuracy_mean": (
        mlp_metric_summary.loc[
            "mean",
            "balanced_accuracy",
        ]
    ),
    "balanced_accuracy_fold_std": (
        mlp_metric_summary.loc[
            "std",
            "balanced_accuracy",
        ]
    ),
    "macro_f1_mean": (
        mlp_mean_macro_f1
    ),
    "macro_f1_fold_std": (
        mlp_metric_summary.loc[
            "std",
            "macro_f1",
        ]
    ),
    "gain_vs_hgb_mean": (
        mlp_mean_macro_f1
        - HGB_MEAN_MACRO_F1
    ),
    "folds_better_than_hgb": (
        mlp_folds_better_than_hgb
    ),
    "competitive_for_seed_repeat": (
        mlp_is_competitive
    ),
    "passes_replacement_gate": (
        mlp_passes_replacement_gate
    ),
})

display(
    mlp_fold_metrics.round(4)
)

display(
    mlp_summary
    .to_frame("value")
    .round(4)
)

print(
    "Next action:",
    (
        "repeat seeds 43 and 44"
        if mlp_is_competitive
        else "stop after seed 42"
    ),
)

,outer_fold,accuracy,balanced_accuracy,macro_f1,hgb_macro_f1,mlp_minus_hgb
0,1,0.7832,0.7832,0.7844,0.7852,-0.0008
1,2,0.6728,0.6743,0.6726,0.6895,-0.0169
2,3,0.7922,0.7935,0.7916,0.7887,0.0029
3,4,0.6992,0.7003,0.6957,0.7159,-0.0202
4,5,0.7311,0.7327,0.7261,0.7434,-0.0173


,value
accuracy_mean,0.7357
accuracy_fold_std,0.051849
balanced_accuracy_mean,0.736796
balanced_accuracy_fold_std,0.051562
macro_f1_mean,0.734076
macro_f1_fold_std,0.052806
gain_vs_hgb_mean,-0.010524
folds_better_than_hgb,1
competitive_for_seed_repeat,False
passes_replacement_gate,False


Next action: stop after seed 42


## 6. Findings and decision

The fixed feature-based MLP selects between 19 and 36 training epochs across
the five nested subject folds. None of the inner runs reaches the 100-epoch
limit, indicating that the early-stopping protocol provides a practical
training duration for every fold.

At seed 42, the MLP achieves:

- mean accuracy: **0.7357**;
- mean balanced accuracy: **0.7368**;
- mean macro-F1: **0.7341 ± 0.0528**.

This is 0.0105 macro-F1 below histogram gradient boosting. The MLP slightly
outperforms boosting on fold 3, by 0.0029, but performs worse on the other four
folds. Its largest deficits occur in folds 4, 5, and 2.

The MLP result is also 0.0005 below the predeclared competitiveness threshold
of 0.7346. Seeds 43 and 44 are therefore not evaluated. This follows the
declared continuation rule and avoids tuning or extending a model after
observing outer-fold performance.

The result remains informative: the MLP performs better on average than the
previous engineered Random Forest and substantially better than logistic
regression, showing that learned nonlinear feature interactions are useful.
However, histogram gradient boosting uses the same engineered representation
more effectively and remains more consistent across held-out subject folds.

Histogram gradient boosting therefore remains the current development leader
at **0.7446 ± 0.0432 macro-F1**. This experiment does not establish that every
possible MLP would underperform, but the fixed architecture provides no
evidence that additional feature-MLP complexity is justified.

The reserved test set remains untouched.